# Mini Project 4 — Steering Vector Lab

Extract a contrastive activation steering vector and inject it at inference. See `03-Mini-Projects/Mini-4-Steering-Vector-Lab.md` for full context.

Pick a contrast: 'cautious & uncertainty-acknowledging' vs 'confident & definitive'. We use Qwen2-1.5B-Instruct as a balance of fitting on a single 16GB GPU (or CPU with patience) and showing a real behavioral effect.

In [ ]:
# !pip install transformer_lens

import torch
from transformer_lens import HookedTransformer

MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()

In [ ]:
CAUTIOUS_PROMPTS = [
    "I'm not entirely sure, but it seems",
    "It is possible that",
    "I might be wrong, but",
    "My best guess is",
    # ... aim for ~30 of these
]
CONFIDENT_PROMPTS = [
    "It is definitely the case that",
    "Without any doubt,",
    "The clear and correct answer is",
    "There is no question that",
    # ... aim for ~30 of these
]
LAYER = model.cfg.n_layers // 2   # middle layer is a reasonable default
HOOK  = f"blocks.{LAYER}.hook_resid_post"

def mean_residual(prompts):
    acts = []
    for p in prompts:
        toks = model.to_tokens(p)
        with torch.no_grad():
            _, cache = model.run_with_cache(toks, names_filter=HOOK)
        acts.append(cache[HOOK][0, -1])  # last token, residual stream
    return torch.stack(acts).mean(0)

cautious_mean  = mean_residual(CAUTIOUS_PROMPTS)
confident_mean = mean_residual(CONFIDENT_PROMPTS)
steering_vec   = cautious_mean - confident_mean
steering_vec   = steering_vec / steering_vec.norm()  # unit-norm; coefficient does the scaling
print("steering_vec shape:", steering_vec.shape)

In [ ]:
def make_steering_hook(vec, coeff):
    def hook(activation, hook_obj):
        return activation + coeff * vec
    return hook

def generate_with_steering(prompt: str, coeff: float, max_new: int = 80) -> str:
    toks = model.to_tokens(prompt)
    if coeff == 0:
        out = model.generate(toks, max_new_tokens=max_new, do_sample=False)
    else:
        with model.hooks(fwd_hooks=[(HOOK, make_steering_hook(steering_vec, coeff))]):
            out = model.generate(toks, max_new_tokens=max_new, do_sample=False)
    return model.tokenizer.decode(out[0])

In [ ]:
TEST_PROMPT = "Is it true that all swans are white?\nAnswer:"
for coeff in [-3, -1, 0, 1, 3]:
    print(f"\n--- coeff = {coeff:+d} ---")
    print(generate_with_steering(TEST_PROMPT, coeff))

## What to do next

1. Take 30 distractor-heavy questions from your main project. Run them through coeff ∈ {-2, -1, 0, +1, +2}.
2. Plot accuracy vs coeff. Also plot accuracy on 30 *clean* questions (general-capability sanity check).
3. The interesting finding looks like: reliability rises with positive coeff *up to a point*, then capability collapses. Find that point.
4. Write a paragraph for your blog post: "Mitigation 5 — Activation Steering."